In [0]:
%pip install xgboost

In [0]:
import mlflow
import xgboost as xgb
import pandas as pd
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType
import datetime

# --- STEP 1: LOAD MODEL & EXTRACT BYTES ---
model_uri = 'runs:/3ab3188cc71f454782c70ea2f2f3e235/model'
print("Loading model on Driver...")

# Load the wrapper (XGBClassifier)
wrapper_model = mlflow.xgboost.load_model(model_uri)

# Extract the underlying native Booster to get access to .save_raw()
# This is the bridge between the sklearn-style wrapper and the native byte format
native_booster = wrapper_model.get_booster()
model_bytes = native_booster.save_raw()

# --- STEP 2: SETUP DATA & SCHEMA ---
gold_df = spark.table("stocks.gold.stocks_w_prev_returns").filter(
    col("Date") >= '2026-01-17'#datetime.datetime.now().date()
)

target_col = 'label'
id_cols = ['Date', 'company', 'feat_highest_return', 'feat_lowest_return', 'feat_daily_return', 'label']
feature_cols = [c for c in gold_df.columns if c not in [target_col] + id_cols]

schema = StructType([
    StructField("prediction", IntegerType(), True),
    StructField("certainty", DoubleType(), True)
])

# --- STEP 3: DEFINE UDF ---
@pandas_udf(schema)
def predict_bytes_udf(iterator: Iterator[Tuple[pd.Series, ...]]) -> Iterator[pd.DataFrame]:
    # INITIALIZATION (Runs once per worker)
    # Reconstruct the native Booster from bytes
    booster = xgb.Booster()
    booster.load_model(bytearray(model_bytes))
    
    # PROCESSING LOOP
    for args in iterator:
        X = pd.concat(args, axis=1)
        X.columns = feature_cols
        
        # Use DMatrix for native prediction
        dtest = xgb.DMatrix(X)
        
        # Native Booster returns probabilities by default
        probs = booster.predict(dtest)
        preds = (probs > 0.5).astype(int)
        
        yield pd.DataFrame({
            'prediction': preds,
            'certainty': probs
        })

# --- STEP 4: EXECUTE ---
gold_df_scored = gold_df.withColumn(
    'model_output', 
    predict_bytes_udf(*[col(c) for c in feature_cols])
)

final_df = gold_df_scored.select(
    "Date", 
    "company", 
    "model_output.prediction", 
    "model_output.certainty",
    "label"
)


In [0]:

final_df.display()

In [0]:

from delta.tables import DeltaTable

if not spark.catalog.tableExists(f"stocks.reporting.predictions"):
    final_df.write.mode("overwrite").format("delta").saveAsTable(f"stocks.reporting.predictions"
        )

delta_table = DeltaTable.forName(spark, f"stocks.reporting.predictions")

delta_table.alias("t").merge(
    final_df.alias("s"),
    "s.Date = t.Date AND s.company = t.company"
).whenNotMatchedInsertAll().execute()


# 2. Load the NEW Source (The reference data)
source_df = spark.table("stocks.gold.stocks_w_prev_returns")

# 3. Execute the Merge
delta_table.alias("p") \
  .merge(
    source_df.alias("s"),
    condition = "p.Date = s.Date AND p.company = s.company"
  ) \
  .whenMatchedUpdate(
    # Update 'label' in predictions using the value from 'gold'
    set = { "label": col("s.label") }
  ) \
  .execute()